# 使用 MyScale 进行嵌入搜索

本 notebook 将带你完成一个简单的流程：下载一些数据，对其进行嵌入，然后使用选定的向量数据库对其建立索引并进行搜索。对于希望在安全环境中存储和搜索我们的嵌入及其自有数据、以支持聊天机器人、主题建模等生产用例的客户来说，这是一个常见需求。

### 什么是向量数据库

向量数据库是专门用于存储、管理和搜索嵌入向量的数据库。近年来，由于 AI 在解决涉及自然语言、图像识别和其他非结构化数据形式的用例方面越来越有效，使用嵌入将非结构化数据（文本、音频、视频等）编码为向量、供机器学习模型使用的做法呈爆发式增长。向量数据库已成为企业交付和扩展这些用例的有效解决方案。

### 为什么使用向量数据库

向量数据库让企业能够把我们在本仓库中分享的许多嵌入用例（例如问答、聊天机器人和推荐服务）用在安全、可扩展的环境中。我们的许多客户能用嵌入在小规模上解决问题，但性能和安全性阻碍了他们投入生产——我们认为向量数据库是解决这一问题的关键组件。在本指南中，我们将介绍对文本数据进行嵌入、将其存储到向量数据库中并用于语义搜索的基础知识。


### 演示流程
演示流程如下：
- **环境设置**：导入包并设置所需的变量
- **加载数据**：加载数据集并使用 OpenAI 嵌入对其进行嵌入
- **MyScale**
    - *环境设置*：设置 MyScale Python 客户端。更多详情请参阅[这里](https://docs.myscale.com/en/python-client/)
    - *索引数据*：我们将创建一张表并为 __content__ 建立索引。
    - *搜索数据*：带着不同的目标运行几个示例查询。

完成本 notebook 后，你应该对如何设置和使用向量数据库有了基本的了解，并可以继续探索使用我们的嵌入的更复杂用例。

## 环境设置

导入所需的库，并设置我们要使用的嵌入模型。

In [ ]:
# We'll need to install the MyScale client
!pip install clickhouse-connect

#Install wget to pull zip file
!pip install wget

In [2]:
import openai

from typing import List, Iterator
import pandas as pd
import numpy as np
import os
import wget
from ast import literal_eval

# MyScale's client library for Python
import clickhouse_connect

# I've set this to our new embeddings model, this can be changed to the embedding model of your choice
EMBEDDING_MODEL = "text-embedding-3-small"

# Ignore unclosed SSL socket warnings - optional in case you get these errors
import warnings

warnings.filterwarnings(action="ignore", message="unclosed", category=ResourceWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning) 

## 加载数据

在本节中，我们将加载在本次会话之前准备好的已嵌入数据。

In [ ]:
embeddings_url = 'https://cdn.openai.com/API/examples/data/vector_database_wikipedia_articles_embedded.zip'

# The file is ~700 MB so this will take some time
wget.download(embeddings_url)

In [ ]:
import zipfile
with zipfile.ZipFile("vector_database_wikipedia_articles_embedded.zip","r") as zip_ref:
    zip_ref.extractall("../data")

In [3]:
article_df = pd.read_csv('../data/vector_database_wikipedia_articles_embedded.csv')

In [4]:
article_df.head()

,id,url,title,text,title_vector,content_vector,vector_id
0,1,https://simple.wikipedia.org/wiki/April,April,April is the fourth month of the year in the J...,"[0.001009464613161981, -0.020700545981526375, ...","[-0.011253940872848034, -0.013491976074874401,...",0
1,2,https://simple.wikipedia.org/wiki/August,August,August (Aug.) is the eighth month of the year ...,"[0.0009286514250561595, 0.000820168002974242, ...","[0.0003609954728744924, 0.007262262050062418, ...",1
2,6,https://simple.wikipedia.org/wiki/Art,Art,Art is a creative activity that expresses imag...,"[0.003393713850528002, 0.0061537534929811954, ...","[-0.004959689453244209, 0.015772193670272827, ...",2
3,8,https://simple.wikipedia.org/wiki/A,A,A or a is the first letter of the English alph...,"[0.0153952119871974, -0.013759135268628597, 0....","[0.024894846603274345, -0.022186409682035446, ...",3
4,9,https://simple.wikipedia.org/wiki/Air,Air,Air refers to the Earth's atmosphere. Air is a...,"[0.02224554680287838, -0.02044147066771984, -0...","[0.021524671465158463, 0.018522677943110466, -...",4


In [5]:
# Read vectors from strings back into a list
article_df['title_vector'] = article_df.title_vector.apply(literal_eval)
article_df['content_vector'] = article_df.content_vector.apply(literal_eval)

# Set vector_id to be a string
article_df['vector_id'] = article_df['vector_id'].apply(str)

In [6]:
article_df.info(show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   id              25000 non-null  int64 
 1   url             25000 non-null  object
 2   title           25000 non-null  object
 3   text            25000 non-null  object
 4   title_vector    25000 non-null  object
 5   content_vector  25000 non-null  object
 6   vector_id       25000 non-null  object
dtypes: int64(1), object(6)
memory usage: 1.3+ MB


## MyScale
接下来我们要介绍的向量数据库是 [MyScale](https://myscale.com)。

[MyScale](https://myscale.com) 是一个基于 Clickhouse 构建的数据库，它将向量搜索与 SQL 分析相结合，提供高性能、精简且完全托管的体验。它旨在支持对结构化数据和向量数据进行联合查询和分析，并为所有数据处理提供全面的 SQL 支持。

使用 [MyScale Console](https://console.myscale.com)，两分钟内即可在你的集群上部署并用 SQL 执行向量搜索。

### 连接到 MyScale

按照[连接详情](https://docs.myscale.com/en/cluster-management/)一节，从 MyScale 控制台获取集群主机、用户名和密码信息，并用它们创建到集群的连接，如下所示：

In [8]:
# initialize client
client = clickhouse_connect.get_client(host='YOUR_CLUSTER_HOST', port=8443, username='YOUR_USERNAME', password='YOUR_CLUSTER_PASSWORD')

### 索引数据

我们将在 MyScale 中创建一张名为 `articles` 的 SQL 表来存储嵌入数据。该表将包含一个使用余弦距离度量的向量索引，以及一个限制嵌入长度的约束。使用以下代码创建 articles 表并插入数据：

In [9]:
# create articles table with vector index
embedding_len=len(article_df['content_vector'][0]) # 1536

client.command(f"""
CREATE TABLE IF NOT EXISTS default.articles
(
    id UInt64,
    url String,
    title String,
    text String,
    content_vector Array(Float32),
    CONSTRAINT cons_vector_len CHECK length(content_vector) = {embedding_len},
    VECTOR INDEX article_content_index content_vector TYPE HNSWFLAT('metric_type=Cosine')
)
ENGINE = MergeTree ORDER BY id
""")

# insert data into the table in batches
from tqdm.auto import tqdm

batch_size = 100
total_records = len(article_df)

# we only need subset of columns
article_df = article_df[['id', 'url', 'title', 'text', 'content_vector']]

# upload data in batches
data = article_df.to_records(index=False).tolist()
column_names = article_df.columns.tolist()

for i in tqdm(range(0, total_records, batch_size)):
    i_end = min(i + batch_size, total_records)
    client.insert("default.articles", data[i:i_end], column_names=column_names)

  0%|          | 0/250 [00:00<?, ?it/s]

由于向量索引会在后台自动构建，在继续搜索之前，我们需要检查它的构建状态。

In [10]:
# check count of inserted data
print(f"articles count: {client.command('SELECT count(*) FROM default.articles')}")

# check the status of the vector index, make sure vector index is ready with 'Built' status
get_index_status="SELECT status FROM system.vector_indices WHERE name='article_content_index'"
print(f"index build status: {client.command(get_index_status)}")

articles count: 25000
index build status: InProgress


### 搜索数据

在 MyScale 中建立索引后，我们就可以执行向量搜索来查找相似内容。首先，我们使用 OpenAI API 为查询生成嵌入。然后，使用 MyScale 执行向量搜索。

In [11]:
query = "Famous battles in Scottish history"

# creates embedding vector from user query
embed = openai.Embedding.create(
    input=query,
    model="text-embedding-3-small",
)["data"][0]["embedding"]

# query the database to find the top K similar content to the given query
top_k = 10
results = client.query(f"""
SELECT id, url, title, distance(content_vector, {embed}) as dist
FROM default.articles
ORDER BY dist
LIMIT {top_k}
""")

# display results
for i, r in enumerate(results.named_results()):
    print(i+1, r['title'])

1 Battle of Bannockburn
2 Wars of Scottish Independence
3 1651
4 First War of Scottish Independence
5 Robert I of Scotland
6 841
7 1716
8 1314
9 1263
10 William Wallace
